# 21.4 结构化流处理 / Structured Streaming (Event Time, Windows, Watermarks)

**中文**:前面处理的都是**有界的、静止的**数据(批处理:一次读完一个文件)。但现实中大量数据是**无界的、持续到达的流**——用户点击、传感器读数、交易、日志,每秒源源不断。**流处理(stream processing)** 要在数据到达时**持续、增量地**计算结果(如"每 5 分钟的实时成交量"),而不是等数据"读完"(它永远读不完)。Spark **Structured Streaming** 的革命性思想是:*把一个数据流看成一张"无限增长、不断追加新行的表"*,于是**流式计算和批处理用同一套 API**。本节从零实现一个 mini 流引擎,亲手攻克流处理最核心也最难的两个概念——**事件时间窗口(event-time windowing)** 和 **水位线(watermark)处理迟到数据**。这是实时数据岗和系统设计面试的必考点。
**English**: So far we processed **bounded, static** data (batch: read a whole file once). But much real-world data is an **unbounded, continuously-arriving stream** — user clicks, sensor readings, transactions, logs, pouring in every second. **Stream processing** must compute results **continuously and incrementally** as data arrives (e.g., "real-time volume every 5 minutes"), rather than waiting for the data to "finish loading" (it never does). Spark **Structured Streaming**'s revolutionary idea: *treat a stream as an "infinitely growing table to which new rows are continuously appended,"* so **streaming and batch use the same API**. This section builds a mini streaming engine from scratch to conquer the two hardest core concepts — **event-time windowing** and **watermarks for late data**. These are must-know for real-time data roles and system-design interviews.

---

**中文**:**事件时间 vs 处理时间(面试必问的区分)**:
**English**: **Event time vs processing time (a must-ask distinction)**:
- **中文**:**事件时间(event time)**:事件**实际发生**的时间(数据里自带的时间戳,如用户点击那一刻)。
  **Event time**: when the event **actually happened** (the timestamp carried in the data, e.g. the moment of a click).
- **中文**:**处理时间(processing time)**:系统**收到并处理**该事件的时间。
  **Processing time**: when the system **received and processed** the event.
- **中文**:两者常**不一致**!手机断网,你的点击(事件时间 10:00)可能 10:05 才传到服务器(处理时间)。分析必须按**事件时间**(否则"每小时统计"会错乱),但事件时间意味着数据**可能乱序、迟到**到达——这就是流处理的核心难题。
  **They often differ!** With a phone offline, your click (event time 10:00) may reach the server at 10:05 (processing time). Analytics must use **event time** (else "hourly stats" break), but event time means data may arrive **out-of-order and late** — the core challenge of streaming.

**中文**:**水位线(watermark)是解决迟到数据的天才设计**。问题:按事件时间开窗统计"[10:00,10:05) 的成交量",什么时候能确定这个窗口"算完了"、可以输出?——如果永远等待迟到数据,窗口永远关不了、状态无限膨胀。**Watermark** 给出答案:*"我认为不会再有比 `watermark = 已见最大事件时间 − 允许迟到时长` 更早的数据了。"* 于是:
**English**: **The watermark is a brilliant design for handling late data.** Problem: when aggregating "volume in [10:00,10:05)" by event time, when can we decide the window is "done" and emit it? — waiting forever for late data means windows never close and state grows unbounded. The **watermark** answers: *"I assume no data earlier than `watermark = max event time seen − allowed lateness` will arrive anymore."* So:
- **中文**:窗口结束时间 < watermark → **窗口定案输出**,并清理其状态。
  Window end < watermark → **finalize and emit the window**, and clean up its state.
- **中文**:**轻微乱序**(在允许迟到时长内)→ 仍能正确归入窗口;**极端迟到**(窗口已定案)→ **丢弃**。这是**延迟 vs 完整性**的权衡:允许迟到越久,结果越完整,但输出越慢、状态越大。
  **Mild out-of-order** (within allowed lateness) → still correctly assigned to its window; **extreme lateness** (window already finalized) → **dropped**. This is the **latency vs completeness** tradeoff: longer allowed lateness = more complete results but slower output and larger state.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 实时/系统设计必考）**
> **中文**:**Structured Streaming 模型**:流=无限追加的表, 流批同一套 API, 增量微批(micro-batch)计算。**事件时间 vs 处理时间**:分析要用事件时间(数据自带戳), 但会乱序/迟到。**窗口**:tumbling(不重叠)/sliding(滑动重叠)/session(会话)。**Watermark**=已见最大事件时间−允许迟到→决定窗口何时定案+丢弃过迟数据+限制状态大小; 核心是**延迟 vs 完整性**权衡。**输出模式(output mode)**:append(只加新结果, 需 watermark)/update(更新变化的行)/complete(全量重输, 用于聚合)。**Exactly-once 恰好一次**:靠 **checkpoint(记录进度/offset)+ 幂等或事务型 sink + 可重放 source(如 Kafka)**。**有状态操作**:窗口聚合、去重、流-流 join 都要维护状态(watermark 限制状态无限增长)。**触发器 trigger**:微批间隔 / 连续处理。**vs 批**:批=有界一次算完, 流=无界持续增量。用途:实时大盘、风控、监控告警、特征实时更新。面试金句:*"Structured Streaming 把流当无限表, 流批同 API; 难点是按事件时间开窗时处理乱序迟到数据——watermark=最大事件时间−允许迟到, 决定窗口何时定案、丢弃过迟数据、限制状态; 恰好一次靠 checkpoint+幂等 sink+可重放 source。"*
> **English**: **Structured Streaming model**: stream = infinitely-appended table, same API for stream & batch, incremental micro-batch computation. **Event vs processing time**: analytics needs event time (timestamp in data), but data is out-of-order/late. **Windows**: tumbling (non-overlapping) / sliding (overlapping) / session. **Watermark** = max event time seen − allowed lateness → decides when a window finalizes + drops too-late data + bounds state size; the core is the **latency vs completeness** tradeoff. **Output modes**: append (only add new results, needs watermark) / update (update changed rows) / complete (re-output everything, for aggregations). **Exactly-once**: via **checkpoint (records progress/offsets) + idempotent or transactional sink + replayable source (e.g. Kafka)**. **Stateful ops**: window aggregation, deduplication, stream-stream joins all maintain state (watermark bounds unbounded growth). **Triggers**: micro-batch interval / continuous processing. **vs batch**: batch = bounded, computed once; stream = unbounded, continuous & incremental. Uses: real-time dashboards, risk control, monitoring/alerting, real-time feature updates. Interview line: *"Structured Streaming treats a stream as an infinite table with the same API as batch; the hard part is handling out-of-order/late data when windowing by event time — the watermark = max event time − allowed lateness decides when windows finalize, drops too-late data, and bounds state; exactly-once relies on checkpoint + idempotent sink + replayable source."*


In [ ]:

# ============================================================
# 从零实现 mini 流引擎:事件时间窗口 + 水位线处理迟到数据 / mini streaming engine
# 中文:造一个事件流(每个事件自带 event_time), 模拟真实的"近乎有序但有乱序 + 少数极端迟到"的到达顺序。
#      按 5 秒的滚动窗口(tumbling)统计计数, 用 watermark 决定窗口何时定案、迟到多少要丢弃。
# English: an event stream (each with its event_time), simulating realistic "nearly-ordered with jitter + a few
#      extreme stragglers" arrival. Count per 5-second tumbling window; use a watermark to finalize windows and drop late data.
# ============================================================
import random, matplotlib.pyplot as plt
random.seed(3)
W=5; ALLOWED_LATENESS=2                                    # 窗口大小5秒, 允许迟到2秒 / window 5s, lateness 2s
events=[]
for t in range(20):                                        # 真实事件, 事件时间 0..19 / true events, event_time 0..19
    for _ in range(random.randint(1,3)): events.append({"event_time":t})
true_counts={}                                             # 离线"真值"作对照 / offline ground truth
for e in events: true_counts[(e["event_time"]//W)*W]=true_counts.get((e["event_time"]//W)*W,0)+1

# 到达顺序:近乎按事件时间 + 轻微乱序; 再追加几个极端迟到者(窗口早已过去)/ arrival order
arrival=sorted(events, key=lambda e: e["event_time"]+random.uniform(-1.2,1.2))   # 局部乱序 / local jitter
stragglers=[{"event_time":3},{"event_time":6},{"event_time":2}]                  # 极端迟到 / extreme late
arrival=arrival+stragglers                                                        # 在最后才到达 / arrive at the very end

win=lambda t:(t//W)*W                                       # 事件属于哪个窗口 / which window
state={}; emitted=set(); watermark=-1; dropped=0; finalized=[]; wm_trace=[]
BATCH=6                                                     # 每个微批处理6个事件 / micro-batch of 6
for i in range(0,len(arrival),BATCH):
    batch=arrival[i:i+BATCH]
    watermark=max(watermark, max(e["event_time"] for e in batch)-ALLOWED_LATENESS)  # 推进水位线 / advance watermark
    wm_trace.append(watermark)
    for e in batch:
        ws=win(e["event_time"])
        if ws in emitted: dropped+=1; continue              # 窗口已定案 → 该事件太迟, 丢弃 / too late → drop
        state[ws]=state.get(ws,0)+1                          # 否则计入窗口状态 / else add to window state
    for ws in sorted(state):                                # 窗口整体低于水位线 → 定案输出 / finalize windows below watermark
        if ws+W <= watermark and ws not in emitted:
            finalized.append((ws,state[ws])); emitted.add(ws)
for ws in sorted(state):                                    # 流结束, 冲刷剩余窗口 / flush remaining at end
    if ws not in emitted: finalized.append((ws,state[ws]))

print(f"{'窗口 window':>14}{'流式计数':>9}{'真值':>7}{'':>4}")
for ws,c in sorted(finalized):
    t=true_counts.get(ws,0); print(f"  [{ws:2},{ws+W:2})        {c:>7}{t:>7}   {'✓ 正确' if c==t else '✗'}")
print(f"\n因超过 watermark 被丢弃的极端迟到事件 / extreme-late events dropped: {dropped}")
print("→ 轻微乱序在 watermark 容忍内被正确计数; 极端迟到(窗口已定案)被丢弃→状态不会无限膨胀, 流不会永远等")


In [ ]:

# ============================================================
# 可视化:事件时间线、窗口、水位线推进 / event timeline, windows, watermark advance
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 窗口计数对比 / window counts
wins=[f"[{ws},{ws+W})" for ws,_ in sorted(finalized)]; cnts=[c for _,c in sorted(finalized)]
ax[0].bar(wins,cnts,color="#4C72B0")
for i,c in enumerate(cnts): ax[0].text(i,c+0.2,c,ha="center",fontsize=11,weight="bold")
ax[0].set_title("事件时间滚动窗口计数(全部正确)"); ax[0].set_ylabel("计数 count"); ax[0].set_xlabel("窗口(事件时间)")
# ② 水位线推进 vs 微批 / watermark advancing over micro-batches
ax[1].plot(range(len(wm_trace)),wm_trace,"o-",color="#C44E52",lw=2,label="watermark(=最大事件时间−2)")
for wend in [5,10,15,20]:
    ax[1].axhline(wend,ls=":",color="gray",alpha=0.5)
    ax[1].text(0,wend+0.1,f"窗口界 {wend}",fontsize=7,color="gray")
ax[1].set_title("水位线随微批推进 → 越过窗口界就定案该窗口"); ax[1].set_xlabel("微批序号 micro-batch"); ax[1].set_ylabel("事件时间")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/big04_viz.png",dpi=80); plt.show()
print("右:watermark 单调上升, 每越过一条窗口界(5/10/15/20)就把对应窗口定案输出并清理状态")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **流处理的本质困难不是"快",而是"时间"**:很多人以为流处理就是"批处理跑得快一点",大错。真正的难点是**事件时间与处理时间的错位**——数据会乱序、会迟到。如果你天真地按"收到的顺序"统计,那么一个网络延迟就会让"10 点的成交"被算进"10 点 05 分的窗口",报表全错。必须按事件时间开窗,而这立刻带来"窗口什么时候算完"的世纪难题。我们的 mini 引擎展示了正确做法:**watermark 让轻微乱序被正确归窗(结果与离线真值完全一致),同时果断丢弃极端迟到者**——否则窗口永远关不了、状态无限膨胀,流最终 OOM 崩溃。
2. **Watermark 是一个"故意的近似",本质是权衡**:watermark 说"我赌不会再有更早的数据了",这个赌注可能赌错(真有超迟到的数据就被丢了)。**允许迟到时长(allowed lateness)** 就是这个赌注的松紧:①设得**长** → 等得久、结果更完整,但**输出延迟高、状态占内存多**;②设得**短** → 输出快、省内存,但**更多迟到数据被丢、结果偏不完整**。没有正确答案,只有业务权衡:实时风控要低延迟(短 watermark),对账报表要完整(长 watermark 或干脆用批处理回补)。**流处理工程师的核心功力,就是调好这个延迟-完整性的旋钮。**
3. **诚实的复杂性:Exactly-once 和状态管理是真正的深水区**。①**恰好一次语义(exactly-once)**:机器会挂、任务会重启,怎么保证"每条数据只被计入一次、不重不漏"?靠三件套:**checkpoint**(持久化处理进度/offset)+ **可重放的 source**(如 Kafka, 挂了能从 offset 重读)+ **幂等/事务型 sink**(重写不会重复)。少一环就做不到 exactly-once。②**状态管理**:窗口聚合、去重、流-流 join 都要在内存/状态后端维护状态,watermark 是防止状态爆炸的关键;状态太大是流作业最常见的故障。③**乱序的现实**很脏:移动端、跨时区、时钟不同步都会制造迟到,watermark 参数几乎总要在生产里反复调。**记住:流处理的 80% 难度在时间语义(事件时间+watermark)和容错(exactly-once),而不在吞吐——把这两点讲清楚,系统设计面试就稳了。**

**English**:
1. **Streaming's essential difficulty isn't "speed" but "time"**: many think stream processing is just "batch, but faster" — badly wrong. The real challenge is the **mismatch between event time and processing time** — data is out-of-order and late. If you naively count by "arrival order," one network delay puts "a 10:00 trade" into the "10:05 window," wrecking the report. You must window by event time, which immediately raises the age-old problem of "when is a window done?" Our mini engine shows the correct approach: **the watermark correctly assigns mild out-of-order data (results exactly match the offline ground truth) while decisively dropping extreme stragglers** — otherwise windows never close, state grows unbounded, and the stream eventually OOMs.
2. **The watermark is a "deliberate approximation," fundamentally a tradeoff**: the watermark bets "no earlier data will arrive," and this bet can be wrong (truly super-late data gets dropped). **Allowed lateness** is how tight that bet is: ① set it **long** → wait longer, more complete results, but **higher output latency and more state in memory**; ② set it **short** → faster output, less memory, but **more late data dropped and less complete results**. There's no right answer, only a business tradeoff: real-time risk control wants low latency (short watermark), reconciliation reports want completeness (long watermark or a batch backfill). **A streaming engineer's core skill is tuning this latency-completeness knob.**
3. **Honest complexity: exactly-once and state management are the real deep end**. ① **Exactly-once semantics**: machines crash and tasks restart, so how to guarantee "each record is counted exactly once, no duplicates or losses"? Via the trio: **checkpoint** (persist processing progress/offsets) + **replayable source** (e.g. Kafka, re-read from offset after a crash) + **idempotent/transactional sink** (rewrites don't duplicate). Miss one and you can't achieve exactly-once. ② **State management**: window aggregation, deduplication, stream-stream joins all maintain state in memory/a state backend, and the watermark is key to preventing state explosion; oversized state is the most common streaming-job failure. ③ **Out-of-order reality** is messy: mobile devices, cross-timezone, clock skew all create lateness, and watermark parameters almost always need repeated tuning in production. **Remember: 80% of streaming's difficulty is in time semantics (event time + watermark) and fault tolerance (exactly-once), not throughput — explain these two clearly and the system-design interview is in hand.**

> 💼 **实战视角 / Practical angle**
> **中文**:流处理落地:①**技术选型**:Spark Structured Streaming(流批统一、微批, 生态好)、**Flink**(真正逐事件低延迟、状态管理强, 实时首选)、Kafka Streams(轻量)。②**必配 Kafka**(21.10)做可重放 source。③**关键参数**:窗口类型/大小、`withWatermark("ts", "10 minutes")` 允许迟到、输出模式、触发间隔、checkpoint 目录。④**恰好一次**:checkpoint + 幂等 sink(如 upsert 到数据库/Delta)。⑤**监控**:处理延迟、积压(lag)、状态大小、迟到丢弃率。⑥**Lambda/Kappa 架构**:实时流给近似快结果, 批处理夜里回补精确值(对账)。面试金句:*"流处理难在时间和容错:按事件时间开窗必然遇到乱序迟到, 用 watermark(最大事件时间−允许迟到)决定窗口定案并丢弃过迟数据, 这是延迟vs完整性权衡; 恰好一次靠 checkpoint+可重放source(Kafka)+幂等sink; 低延迟强状态场景选 Flink。"*
> **English**: Streaming in practice: ① **tech choice**: Spark Structured Streaming (unified stream/batch, micro-batch, great ecosystem), **Flink** (true per-event low latency, strong state management, top choice for real-time), Kafka Streams (lightweight). ② **pair with Kafka** (21.10) as a replayable source. ③ **key parameters**: window type/size, `withWatermark("ts", "10 minutes")` allowed lateness, output mode, trigger interval, checkpoint directory. ④ **exactly-once**: checkpoint + idempotent sink (e.g. upsert into a database/Delta). ⑤ **monitoring**: processing latency, backlog (lag), state size, late-drop rate. ⑥ **Lambda/Kappa architecture**: the real-time stream gives fast approximate results, batch backfills exact values overnight (reconciliation). Interview line: *"Streaming's difficulty is time and fault tolerance: event-time windowing inevitably meets out-of-order/late data, handled by a watermark (max event time − allowed lateness) that finalizes windows and drops too-late data, a latency-vs-completeness tradeoff; exactly-once relies on checkpoint + replayable source (Kafka) + idempotent sink; for low-latency, heavy-state cases choose Flink."*

---
### 小结 / Summary
- **中文**:流处理=无界持续增量计算; Structured Streaming 把流当无限表, 流批同一套 API。
- **English**: Stream processing = unbounded, continuous, incremental compute; Structured Streaming treats a stream as an infinite table, same API for stream & batch.
- **中文**:核心难点=事件时间乱序/迟到; watermark(最大事件时间−允许迟到)决定窗口定案+丢过迟数据+限状态, 是延迟-完整性权衡。
- **English**: Core difficulty = out-of-order/late event time; the watermark (max event time − allowed lateness) finalizes windows + drops late data + bounds state, a latency-completeness tradeoff.
- **中文**:恰好一次靠 checkpoint+可重放 source(Kafka)+幂等 sink; 低延迟强状态选 Flink。
- **English**: Exactly-once via checkpoint + replayable source (Kafka) + idempotent sink; for low latency & heavy state choose Flink.
